<a href="https://colab.research.google.com/github/PACIFIK10/rcm-performance-dashboard/blob/main/data_generation/rcm_claims_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install faker psycopg2-binary --quiet

import psycopg2
from faker import Faker
import random
from google.colab import userdata
fake = Faker()
Faker.seed(42)
random.seed(42)

# ============================================
# Supabase connection
# ============================================
conn = psycopg2.connect(
    host="aws-1-us-west-2.pooler.supabase.com",      # session pooler host, NOT db.xxxx.supabase.co
    database="postgres",
    user=userdata.get('SUPABASE_USER'),      # note: project ref appended to username
    password=userdata.get('SUPABASE_PASSWORD'),
    port=5432
)
cur = conn.cursor()

# ============================================
# Specialty distribution
# ============================================
specialty_distribution = {
    'Optometry / Eye Care': 20,
    'Chiropractic Care': 19,
    'Behavioral & Mental Health': 14,
    'Orthopedics & Spine Surgery': 12,
    'Plastic & Reconstructive Surgery': 5
}

def generate_npi():
    # NPIs are 10 digits, first digit is 1 or 2 per CMS convention
    return str(random.choice([1, 2])) + ''.join([str(random.randint(0,9)) for _ in range(9)])

providers = []
for specialty, count in specialty_distribution.items():
    for _ in range(count):
        providers.append((fake.name(), specialty, generate_npi(), 'Physician'))

In [ ]:
import numpy as np

np.random.seed(42)
random.seed(42)

# ============================================
# Payer denial multipliers (Medicare highest, per your R1 experience)
# ============================================
payer_denial_multiplier = {
    'Medicare Part B': 1.35,
    'Medicare Advantage': 1.25,
    'State Medicaid': 1.00,
    'Managed Medicaid': 0.95,
    'BCBS PPO': 0.85,
    'BCBS HMO': 0.90,
    'Commercial - Aetna/UHC': 0.75,
}

# Base specialty denial rates (defined earlier)
specialty_denial_rate = {
    'Chiropractic Care': 0.32,
    'Optometry / Eye Care': 0.28,
    'Behavioral & Mental Health': 0.24,
    'Orthopedics & Spine Surgery': 0.20,
    'Plastic & Reconstructive Surgery': 0.17,
}

# Map payer_id -> payer_name for lookup
payer_id_to_name = dict(zip(payers_df['payer_id'], payers_df['payer_name']))

# ============================================
# Denial reason pool, weighted by category tendency
# (Administrative denials most common/fixable, Clinical hardest)
# ============================================
denial_reasons = {
    'D01': {'category': 'Administrative', 'weight': 0.15},  # Missing prior auth
    'D02': {'category': 'Clinical', 'weight': 0.12},        # Invalid diagnosis code
    'D03': {'category': 'Eligibility', 'weight': 0.13},     # Eligibility lapsed
    'D04': {'category': 'Administrative', 'weight': 0.10},  # Duplicate claim
    'D05': {'category': 'Eligibility', 'weight': 0.10},     # Not covered under plan
    'D06': {'category': 'Clinical', 'weight': 0.10},        # Incorrect CPT/HCPCS
    'D07': {'category': 'Administrative', 'weight': 0.10},  # Timely filing exceeded
    'D08': {'category': 'Eligibility', 'weight': 0.08},     # Coordination of benefits
    'D09': {'category': 'Clinical', 'weight': 0.10},        # Medical necessity
    'D10': {'category': 'Administrative', 'weight': 0.02},  # Missing/invalid NPI
}
reason_codes = list(denial_reasons.keys())
reason_weights = [denial_reasons[r]['weight'] for r in reason_codes]

# ============================================
# Resubmission parameters by category
# ============================================
resubmit_rate_by_category = {'Administrative': 0.60, 'Eligibility': 0.45, 'Clinical': 0.30}
resubmit_success_by_category = {'Administrative': 0.85, 'Eligibility': 0.65, 'Clinical': 0.45}

# ============================================
# Process each claim
# ============================================
statuses, paid_amounts, payment_dates, first_pass_flags = [], [], [], []
denial_records = []
denial_id_counter = 1

for idx, row in claims_df.iterrows():
    payer_name = payer_id_to_name[row['payer_id']]
    base_rate = specialty_denial_rate[row['specialty']]
    multiplier = payer_denial_multiplier[payer_name]
    denial_prob = min(base_rate * multiplier, 0.55)  # cap to keep realistic

    payer_days = int(payers_df.loc[payers_df['payer_name'] == payer_name, 'avg_days_to_pay'].values[0])

    is_denied = np.random.random() < denial_prob

    if not is_denied:
        # Paid on first pass
        paid_amount = row['allowed_amount']
        payment_date = row['submission_date'] + timedelta(days=int(np.random.normal(payer_days, payer_days*0.2)))
        statuses.append('Paid')
        paid_amounts.append(paid_amount)
        payment_dates.append(payment_date)
        first_pass_flags.append(True)
    else:
        # Denied — pick a reason
        reason_code = np.random.choice(reason_codes, p=reason_weights)
        category = denial_reasons[reason_code]['category']
        denial_date = row['submission_date'] + timedelta(days=int(np.random.uniform(10, 20)))

        will_resubmit = np.random.random() < resubmit_rate_by_category[category]

        if will_resubmit:
            denial_records.append({
                'denial_id': denial_id_counter, 'claim_id': row['claim_id'],
                'reason_code': reason_code, 'denial_date': denial_date, 'resubmitted_flag': True
            })
            denial_id_counter += 1

            resubmit_success = np.random.random() < resubmit_success_by_category[category]
            resubmit_date = denial_date + timedelta(days=int(np.random.uniform(15, 30)))

            if resubmit_success:
                paid_amount = row['allowed_amount']
                payment_date = resubmit_date + timedelta(days=int(np.random.normal(payer_days, payer_days*0.2)))
                statuses.append('Paid')
                paid_amounts.append(paid_amount)
                payment_dates.append(payment_date)
                first_pass_flags.append(False)
            else:
                statuses.append('Denied')
                paid_amounts.append(0.0)
                payment_dates.append(None)
                first_pass_flags.append(False)
        else:
            denial_records.append({
                'denial_id': denial_id_counter, 'claim_id': row['claim_id'],
                'reason_code': reason_code, 'denial_date': denial_date, 'resubmitted_flag': False
            })
            denial_id_counter += 1
            statuses.append('Denied')
            paid_amounts.append(0.0)
            payment_dates.append(None)
            first_pass_flags.append(False)

claims_df['claim_status'] = statuses
claims_df['paid_amount'] = paid_amounts
claims_df['payment_date'] = payment_dates
claims_df['first_pass_flag'] = first_pass_flags

denials_df = pd.DataFrame(denial_records)

# ============================================
# Validation output
# ============================================
print(f"✓ Processed {len(claims_df)} claims\n")
print("Claim status breakdown:")
print(claims_df['claim_status'].value_counts())
print(f"\nFirst-pass resolution rate: {claims_df['first_pass_flag'].mean():.1%}")
print(f"\nOverall denial rate (any denial recorded): {len(denials_df) / len(claims_df):.1%}")

print(f"\nDenial rate by specialty:")
denied_by_specialty = claims_df.groupby('specialty').apply(lambda x: (x['claim_status']=='Denied').mean() +
    (x['claim_id'].isin(denials_df['claim_id'])).mean() - (x['claim_status']=='Denied').mean())
print(claims_df.groupby('specialty')['claim_id'].apply(
    lambda ids: denials_df['claim_id'].isin(ids).sum() / len(ids)
).sort_values(ascending=False))

print(f"\nDenial rate by payer:")
claims_with_payer = claims_df.merge(payers_df[['payer_id','payer_name']], on='payer_id')
print(claims_with_payer.groupby('payer_name')['claim_id'].apply(
    lambda ids: denials_df['claim_id'].isin(ids).sum() / len(ids)
).sort_values(ascending=False))

print(f"\nResubmission stats:")
print(f"  Total initial denials: {len(denials_df)}")
print(f"  Resubmitted: {denials_df['resubmitted_flag'].sum()} ({denials_df['resubmitted_flag'].mean():.1%})")
print(f"  Eventually paid after resubmission: {(claims_df['first_pass_flag']==False) & (claims_df['claim_status']=='Paid')}".count(True) if False else claims_df[(claims_df['first_pass_flag']==False) & (claims_df['claim_status']=='Paid')].shape[0])

✓ Processed 119677 claims

Claim status breakdown:
claim_status
Paid      97262
Denied    22415
Name: count, dtype: int64

First-pass resolution rate: 72.2%

Overall denial rate (any denial recorded): 27.8%

Denial rate by specialty:
specialty
Chiropractic Care                   0.323398
Optometry / Eye Care                0.284989
Behavioral & Mental Health          0.244316
Orthopedics & Spine Surgery         0.205010
Plastic & Reconstructive Surgery    0.172875
Name: claim_id, dtype: float64

Denial rate by payer:
payer_name
Medicare Part B           0.375826
Medicare Advantage        0.340345
State Medicaid            0.278210
Managed Medicaid          0.259083
BCBS HMO                  0.250793
BCBS PPO                  0.232166
Commercial - Aetna/UHC    0.206161
Name: claim_id, dtype: float64

Resubmission stats:
  Total initial denials: 33217
  Resubmitted: 15256 (45.9%)
10802


/tmp/ipykernel_1452/3717597432.py:139: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  denied_by_specialty = claims_df.groupby('specialty').apply(lambda x: (x['claim_status']=='Denied').mean() +


In [ ]:
import pandas as pd
import numpy as np

# ============================================
# Step 1: Merge denial info onto claims to get a resolution_date
# ============================================
claims_df = claims_df.merge(
    denials_df[['claim_id', 'denial_date']], on='claim_id', how='left'
)

def compute_resolution_date(row):
    if row['claim_status'] == 'Paid':
        return row['payment_date']
    else:
        # Denied (final) — resolved at denial_date if never resubmitted,
        # or ~30 days after denial_date if resubmitted but ultimately failed
        if pd.notna(row['denial_date']):
            return row['denial_date'] + timedelta(days=30)
        return row['submission_date'] + timedelta(days=20)  # fallback safety

claims_df['resolution_date'] = claims_df.apply(compute_resolution_date, axis=1)

print("✓ Resolution dates computed")
print(claims_df[['claim_id','submission_date','claim_status','payment_date','denial_date','resolution_date']].head())

# ============================================
# Step 2: Generate AR_Snapshot — one row per claim per month-end
# it remained outstanding (submission_date <= month_end < resolution_date)
# ============================================
def aging_bucket(days):
    if days <= 30: return '0-30'
    elif days <= 60: return '31-60'
    elif days <= 90: return '61-90'
    else: return '90+'

month_ends = pd.date_range(START_DATE, END_DATE, freq='M')

ar_records = []
snapshot_id = 1

for _, row in claims_df.iterrows():
    sub_date = pd.Timestamp(row['submission_date'])
    res_date = pd.Timestamp(row['resolution_date'])

    for month_end in month_ends:
        if sub_date <= month_end < res_date:
            days_out = (month_end - sub_date).days
            ar_records.append({
                'snapshot_id': snapshot_id,
                'claim_id': row['claim_id'],
                'snapshot_date': month_end.date(),
                'days_outstanding': days_out,
                'aging_bucket': aging_bucket(days_out)
            })
            snapshot_id += 1

ar_snapshot_df = pd.DataFrame(ar_records)
print(f"\n✓ Generated {len(ar_snapshot_df)} AR snapshot rows")
print(ar_snapshot_df['aging_bucket'].value_counts())

# ============================================
# Step 3: Monthly_Forecast — actual collections by month + a simple forecast
# ============================================
claims_df['submission_month'] = pd.to_datetime(claims_df['submission_date']).dt.to_period('M')
actual_monthly = claims_df.groupby('submission_month')['paid_amount'].sum().reset_index()
actual_monthly.columns = ['month', 'actual_collections']
actual_monthly['month'] = actual_monthly['month'].dt.to_timestamp()

# Simple forecast: trailing 3-month moving average, projected forward with
# random noise to simulate realistic forecast error (+/- 8%)
actual_monthly = actual_monthly.sort_values('month').reset_index(drop=True)
actual_monthly['forecasted_collections'] = (
    actual_monthly['actual_collections'].rolling(3, min_periods=1).mean()
    * np.random.uniform(0.92, 1.08, size=len(actual_monthly))
).round(2)
actual_monthly['variance'] = (actual_monthly['actual_collections'] - actual_monthly['forecasted_collections']).round(2)
actual_monthly.insert(0, 'forecast_id', range(1, len(actual_monthly)+1))

monthly_forecast_df = actual_monthly[['forecast_id','month','forecasted_collections','actual_collections','variance']]
print(f"\n✓ Generated {len(monthly_forecast_df)} monthly forecast rows")
print(monthly_forecast_df.head(10))

✓ Resolution dates computed
   claim_id submission_date claim_status payment_date denial_date  \
0         1      2023-08-17         Paid   2023-09-24         NaN   
1         2      2024-07-17       Denied         None  2024-08-04   
2         3      2023-10-13         Paid   2023-11-22         NaN   
3         4      2023-06-28         Paid   2023-08-28  2023-07-16   
4         5      2023-03-03         Paid   2023-03-21         NaN   

  resolution_date  
0      2023-09-24  
1      2024-09-03  
2      2023-11-22  
3      2023-08-28  
4      2023-03-21  


/tmp/ipykernel_1452/3831067832.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_ends = pd.date_range(START_DATE, END_DATE, freq='M')



✓ Generated 164295 AR snapshot rows
aging_bucket
0-30     116395
31-60     42469
61-90      5078
90+         353
Name: count, dtype: int64

✓ Generated 36 monthly forecast rows
   forecast_id      month  forecasted_collections  actual_collections  \
0            1 2023-01-01               628156.82           673572.28   
1            2 2023-02-01               585836.86           575790.68   
2            3 2023-03-01               673481.05           665845.16   
3            4 2023-04-01               670662.95           644831.38   
4            5 2023-05-01               717333.01           684047.14   
5            6 2023-06-01               704012.80           670157.98   
6            7 2023-07-01               662260.90           673198.03   
7            8 2023-08-01               650244.60           696112.90   
8            9 2023-09-01               684338.41           638685.09   
9           10 2023-10-01               629629.63           678230.03   

   variance  
0  4

In [ ]:
import pandas as pd

conn = psycopg2.connect(
    host="aws-1-us-west-2.pooler.supabase.com",
    database="postgres",
    user="postgres.hsopiarfbvkxtadomhmz",
    password="277304@Ballia",
    port=5432
)

tables = ['claim', 'payer', 'provider', 'denial', 'denial_reason', 'ar_snapshot', 'monthly_forecast']

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    df.to_csv(f"{table}.csv", index=False)
    print(f"✓ Exported {table}: {len(df)} rows")

conn.close()

/tmp/ipykernel_1452/3539953455.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)


✓ Exported claim: 119677 rows
✓ Exported payer: 7 rows
✓ Exported provider: 70 rows
✓ Exported denial: 33217 rows
✓ Exported denial_reason: 10 rows
✓ Exported ar_snapshot: 164295 rows
✓ Exported monthly_forecast: 36 rows


In [ ]:
import shutil
shutil.make_archive('rcm_data_export', 'zip', '.', base_dir='.')
from google.colab import files
files.download('rcm_data_export.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="aws-1-us-west-2.pooler.supabase.com",
    database="postgres",
    user="postgres.hsopiarfbvkxtadomhmz",
    password="277304@Ballia",
    port=5432
)

tables = ['payer', 'provider', 'denial_reason', 'claim', 'denial', 'ar_snapshot', 'monthly_forecast']

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    df.to_csv(f"{table}.csv", index=False)
    print(f"✓ Exported {table}.csv: {len(df)} rows, {len(df.columns)} columns")

conn.close()

# Bundle into a single zip for easy download
import shutil
import os

os.makedirs('rcm_export', exist_ok=True)
for table in tables:
    shutil.move(f"{table}.csv", f"rcm_export/{table}.csv")

shutil.make_archive('rcm_data_export', 'zip', 'rcm_export')

from google.colab import files
files.download('rcm_data_export.zip')

/tmp/ipykernel_1452/1956309905.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)


✓ Exported payer.csv: 7 rows, 5 columns
✓ Exported provider.csv: 70 rows, 5 columns
✓ Exported denial_reason.csv: 10 rows, 3 columns


In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import numpy as np
import pandas as pd

conn = psycopg2.connect(
    host="aws-1-us-west-2.pooler.supabase.com",
    database="postgres",
    user="postgres.hsopiarfbvkxtadomhmz",
    password="277304@Ballia",
    port=5432
)
cur = conn.cursor()

def clean_for_insert(df):
    """Replace pandas NaN/NaT with None so psycopg2 writes SQL NULL correctly."""
    return df.astype(object).where(pd.notnull(df), None)

# ============================================
# 1. CLAIM table — trim to exact schema columns, in exact order
# ============================================
claim_insert_df = clean_for_insert(claims_df[[
    'claim_id', 'payer_id', 'provider_id', 'patient_id',
    'submission_date', 'billed_amount', 'allowed_amount',
    'paid_amount', 'payment_date', 'claim_status', 'first_pass_flag'
]])

claim_rows = list(claim_insert_df.itertuples(index=False, name=None))

execute_values(cur, """
    INSERT INTO claim (claim_id, payer_id, provider_id, patient_id,
                        submission_date, billed_amount, allowed_amount,
                        paid_amount, payment_date, claim_status, first_pass_flag)
    VALUES %s
""", claim_rows, page_size=5000)
conn.commit()
print(f"✓ Inserted {len(claim_rows)} claims")

# ============================================
# 2. DENIAL table
# ============================================
denial_insert_df = clean_for_insert(denials_df[[
    'denial_id', 'claim_id', 'reason_code', 'denial_date', 'resubmitted_flag'
]])
denial_rows = list(denial_insert_df.itertuples(index=False, name=None))

execute_values(cur, """
    INSERT INTO denial (denial_id, claim_id, reason_code, denial_date, resubmitted_flag)
    VALUES %s
""", denial_rows, page_size=5000)
conn.commit()
print(f"✓ Inserted {len(denial_rows)} denials")

# ============================================
# 3. AR_SNAPSHOT table
# ============================================
ar_insert_df = clean_for_insert(ar_snapshot_df[[
    'snapshot_id', 'claim_id', 'snapshot_date', 'days_outstanding', 'aging_bucket'
]])
ar_rows = list(ar_insert_df.itertuples(index=False, name=None))

execute_values(cur, """
    INSERT INTO ar_snapshot (snapshot_id, claim_id, snapshot_date, days_outstanding, aging_bucket)
    VALUES %s
""", ar_rows, page_size=5000)
conn.commit()
print(f"✓ Inserted {len(ar_rows)} AR snapshots")

# ============================================
# 4. MONTHLY_FORECAST table
# ============================================
forecast_insert_df = clean_for_insert(monthly_forecast_df[[
    'forecast_id', 'month', 'forecasted_collections', 'actual_collections', 'variance'
]])
forecast_rows = list(forecast_insert_df.itertuples(index=False, name=None))

execute_values(cur, """
    INSERT INTO monthly_forecast (forecast_id, month, forecasted_collections, actual_collections, variance)
    VALUES %s
""", forecast_rows, page_size=5000)
conn.commit()
print(f"✓ Inserted {len(forecast_rows)} monthly forecasts")

# ============================================
# 5. Reset auto-increment sequences to match our explicit IDs
# (we inserted explicit claim_id/denial_id/etc. rather than letting
# SERIAL generate them, so the sequence counters need to catch up —
# otherwise a future INSERT without an explicit ID would collide)
# ============================================
for table, id_col in [('claim','claim_id'), ('denial','denial_id'),
                        ('ar_snapshot','snapshot_id'), ('monthly_forecast','forecast_id')]:
    cur.execute(f"SELECT setval(pg_get_serial_sequence('{table}', '{id_col}'), MAX({id_col})) FROM {table};")
conn.commit()
print("✓ Sequences synced")

cur.close()
conn.close()